<a href="https://colab.research.google.com/github/Ivan07cabrera/Holamundooo/blob/main/Tarea_1__Git_y_documentacion_de_RNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Tarea 1: Git y documentacion de RNA

Implementaremos una red neuronal básica usando la paqueteria de numpy. El código es proveniente del siguiente link: https://github.com/MichalDanielDobrzanski/DeepLearningPython

Del repositorio network.py

"""
network.py: en qué consiste?

~~~~~~~~~~
IT WORKS

A module to implement the stochastic gradient descent learning
algorithm for a feedforward neural network.  Gradients are calculated
using backpropagation.  Note that I have focused on making the code
simple, easily readable, and easily modifiable.  It is not optimized,
and omits many desirable features.
"""

In [3]:
from tensorflow.keras.datasets import mnist
dataset=mnist.load_data()
(x_train, y_train), (x_test, y_test) = dataset

In [4]:
# carga de código  network.py

#### Libraries
# Standard library
import random

# Third-party libraries
import numpy as np

class Network(object):

    def __init__(self, sizes):
        """The list ``sizes`` contains the number of neurons in the
        respective layers of the network.  For example, if the list
        was [2, 3, 1] then it would be a three-layer network, with the
        first layer containing 2 neurons, the second layer 3 neurons,
        and the third layer 1 neuron.  The biases and weights for the
        network are initialized randomly, using a Gaussian
        distribution with mean 0, and variance 1.  Note that the first
        layer is assumed to be an input layer, and by convention we
        won't set any biases for those neurons, since biases are only
        ever used in computing the outputs from later layers."""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x)
                        for x, y in zip(sizes[:-1], sizes[1:])]

    def feedforward(self, a):
        """Return the output of the network if ``a`` is input."""
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a)+b)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta,
            test_data=None):
        """Train the neural network using mini-batch stochastic
        gradient descent.  The ``training_data`` is a list of tuples
        ``(x, y)`` representing the training inputs and the desired
        outputs.  The other non-optional parameters are
        self-explanatory.  If ``test_data`` is provided then the
        network will be evaluated against the test data after each
        epoch, and partial progress printed out.  This is useful for
        tracking progress, but slows things down substantially."""

        training_data = list(training_data)
        n = len(training_data)

        if test_data:
            test_data = list(test_data)
            n_test = len(test_data)

        for j in range(epochs):
            random.shuffle(training_data)   #Mezcla de datos
            mini_batches = [
                training_data[k:k+mini_batch_size]      #Creación de minibatches
                for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)  #entrenamiento de cada minibatch
            if test_data:
                print("Epoch {} : {} / {}".format(j,self.evaluate(test_data),n_test))
            else:
                print("Epoch {} complete".format(j))

    def update_mini_batch(self, mini_batch, eta):
        """Update the network's weights and biases by applying
        gradient descent using backpropagation to a single mini batch.
        The ``mini_batch`` is a list of tuples ``(x, y)``, and ``eta``
        is the learning rate."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]    #inicializa gradientes de los pesos y bias
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)  #Utiliza backpropagation
            nabla_b = [nb+dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw+dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
        self.weights = [w-(eta/len(mini_batch))*nw
                        for w, nw in zip(self.weights, nabla_w)]  #Actualización de pesos y bias
        self.biases = [b-(eta/len(mini_batch))*nb
                       for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x, y):
        """Return a tuple ``(nabla_b, nabla_w)`` representing the
        gradient for the cost function C_x.  ``nabla_b`` and
        ``nabla_w`` are layer-by-layer lists of numpy arrays, similar
        to ``self.biases`` and ``self.weights``."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        # feedforward
        activation = x
        activations = [x] # list to store all the activations, layer by layer
        zs = [] # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation)+b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        # backward pass
        delta = self.cost_derivative(activations[-1], y) * \
            sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())
        # Note that the variable l in the loop below is used a little
        # differently to the notation in Chapter 2 of the book.  Here,
        # l = 1 means the last layer of neurons, l = 2 is the
        # second-last layer, and so on.  It's a renumbering of the
        # scheme in the book, used here to take advantage of the fact
        # that Python can use negative indices in lists.
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())
        return (nabla_b, nabla_w)

    def evaluate(self, test_data):
        """Return the number of test inputs for which the neural
        network outputs the correct result. Note that the neural
        network's output is assumed to be the index of whichever
        neuron in the final layer has the highest activation."""
        test_results = [(np.argmax(self.feedforward(x)), y)
                        for (x, y) in test_data]
        return sum(int(x == y) for (x, y) in test_results)

    def cost_derivative(self, output_activations, y):
        """Return the vector of partial derivatives \partial C_x /
        \partial a for the output activations."""
        return (output_activations-y)

#### Miscellaneous functions
def sigmoid(z):
    """The sigmoid function."""
    return 1.0/(1.0+np.exp(-z))

def sigmoid_prime(z):
    """Derivative of the sigmoid function."""
    return sigmoid(z)*(1-sigmoid(z))

<>:126: SyntaxWarning: invalid escape sequence '\p'
<>:126: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_7312/925615437.py:126: SyntaxWarning: invalid escape sequence '\p'
  """Return the vector of partial derivatives \partial C_x /




*   La primera función nos indica la rquitectura de la red: número de neuronas y el número de capas que tendrá y los biases y pesos son inicializados usando una distribución gausiana con media 0  varianza 1.
*   La segunda función nos indica como "pasarán los datos a través de las capas usando los bias y pesos y datos por una función de activación.


*  La tercera función entrena la red usando el gradiente descendiente estocástico en pequeños conjuntos de datos (minibatches)
*   La cuatra función actualiza los pesos y bias de los minibatcho usando backpropagation


*   La quinta función indica cómo la red “aprende” calculando derivadas capa por capa. (backpropagation)









In [5]:
def vectorized_result(j):
    """Return a 10-dimensional unit vector with a 1.0 in the jth
    position and zeroes elsewhere.  This is used to convert a digit
    (0...9) into a corresponding desired output from the neural
    network."""
    e = np.zeros((10, 1))
    e[j] = 1.0
    return e

In [6]:
#x_train_flattened = [np.reshape(x, (784, 1)) for x in x_train]
x_train_flattened = [np.reshape(x, (784, 1))/255. for x in x_train]
x_test_flattened = [np.reshape(x, (784, 1))/255. for x in x_test]
y_train_categorical = [vectorized_result(y) for y in y_train]

training_data = list(zip(x_train_flattened, y_train_categorical))
test_data = list(zip(x_test_flattened, y_test))

In [7]:
net = Network([784, 30, 10])
net.SGD(training_data, 30, 10, 3.0, test_data=test_data)

Epoch 0 : 9116 / 10000
Epoch 1 : 9285 / 10000
Epoch 2 : 9348 / 10000
Epoch 3 : 9408 / 10000
Epoch 4 : 9411 / 10000
Epoch 5 : 9427 / 10000
Epoch 6 : 9450 / 10000
Epoch 7 : 9437 / 10000
Epoch 8 : 9451 / 10000
Epoch 9 : 9450 / 10000
Epoch 10 : 9465 / 10000
Epoch 11 : 9454 / 10000
Epoch 12 : 9504 / 10000
Epoch 13 : 9490 / 10000
Epoch 14 : 9481 / 10000
Epoch 15 : 9514 / 10000
Epoch 16 : 9484 / 10000
Epoch 17 : 9512 / 10000
Epoch 18 : 9512 / 10000
Epoch 19 : 9507 / 10000
Epoch 20 : 9533 / 10000
Epoch 21 : 9521 / 10000
Epoch 22 : 9512 / 10000
Epoch 23 : 9514 / 10000
Epoch 24 : 9540 / 10000
Epoch 25 : 9523 / 10000
Epoch 26 : 9521 / 10000
Epoch 27 : 9500 / 10000
Epoch 28 : 9526 / 10000
Epoch 29 : 9508 / 10000
